In [ ]:
import requests
import pandas as pd
import numpy as np
import sqlite3 as sql
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

apiKey = "RGAPI-d54fbc39-37bf-4af8-b517-9903d66e77be"

#gets unique ID number for a player from their visable in-game name+tagline or summoner name
def get_puuid(summonerId=None, gameName = None, tagLine=None):
    if summonerId is not None:
        link = 'https://na1.api.riotgames.com/lol/summoner/v4/summoners/'
        response = requests.get(link + summonerId + "?api_key=" + apiKey)
        return response.json()['puuid']
    else:
        link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={apiKey}'
        response = requests.get(link)
        return response.json()['puuid']

In [ ]:
#gets the in-game name+tagline given a puuid
def get_idtag(puuid=None):
    link = 'https://americas.api.riotgames.com/riot/account/v1/accounts/by-puuid'
    response = requests.get(link + puuid + "?api_key=" + apiKey)
    id = {'gameName': response.json()['gameName'], 'tagLine': response.json()['tagLine']}
    return id

In [ ]:
#Gets the lastest (default latest 10) games played by a player from their puuid
def get_match_history(puuid=None, start=0, matches=10):
    link = f'https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?'
    params = f'start={start}&count={matches}'

    response = requests.get(link + params + '&api_key=' + apiKey)
    if response.status_code == 200:
        return response.json()
    else:
        print(f'Error getting match history for {puuid}') 
        return []

# Returns a unique list of matches from the top ranked players returned from get_ladder().
# Gets the last (default 10) matches for each top ranked players before filtering out only unique matchIDs
def get_match_ids(ladder, matches=10):
    all_matches = set()

    for index, player in ladder.iterrows():
        player_puuid = player.get('puuid')
        if not player_puuid:
            continue

        print(f'Fetching match history for player {index + 1}...')
        player_matches = get_match_history(puuid=player_puuid, matches=matches)
        
        for match_id in player_matches:
            all_matches.add(match_id)
        time.sleep(1.2)

    unique_top_matches = list(all_matches)
    top_matches_df = pd.DataFrame(unique_top_matches, columns=['matchId'])
    return top_matches_df

#Returns the json for a specific match from a matchId number from someone's match history
def get_match_data(matchId=None):
    link = f'https://americas.api.riotgames.com'
    param = f'/lol/match/v5/matches/{matchId}'

    response = requests.get(link + param + '?api_key=' + apiKey)
    return response.json()

In [ ]:
#Gets the top players (base top 10 players max unless specified) in all of League of Legends. Top ranks are Master->Grandmaster->Challenger w/ Challenger being the top rank
def get_ladder(top=10):
    root = 'https://na1.api.riotgames.com'
    chall = '/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    gm = '/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = '/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_response = requests.get(root + chall + "?api_key=" + apiKey)
    gm_response = requests.get(root + gm + "?api_key=" + apiKey)
    master_response = requests.get(root + master + "?api_key=" + apiKey)

    if chall_response.status_code != 200 or gm_response.status_code != 200 or master_response.status_code != 200:
        print(f'Error: Ladder Status Codes: {chall_response.status_code}, {gm_response.status_code}, {master_response.status_code}')
        return pd.DataFrame()

    try:
        chall_df = pd.DataFrame(chall_response.json()['entries'])
        gm_df = pd.DataFrame(gm_response.json()['entries'])
        master_df = pd.DataFrame(master_response.json()['entries'])

        solo_queue_df = pd.concat([chall_df, gm_df, master_df]).reset_index(drop=True)
        solo_queue_df = solo_queue_df.sort_values(by='leaguePoints', ascending=False)
        solo_queue_df = solo_queue_df.head(top)
        solo_queue_df = solo_queue_df.drop(columns= ['rank', 'inactive', 'veteran', 'freshBlood'], errors='ignore')
        return solo_queue_df.reset_index(drop=True)
    
    except Exception as e:
        print(f'Json Error: {e}')
        return pd.DataFrame()

# Gives up to 9 players per match not including the seed puuid. Less if people are partied together.
# Return a set of unique average ranked played from the last (default 10) matches played with a given puuid. 
# In other words, returns about matches * 10 puuids
def get_average_players(puuid=None, matches=10):
    seed_puuid = puuid
    if not seed_puuid:
        print('Error: seed_puuid is empty or None')
        return pd.DataFrame(columns=['puuid'])
    
    average_matches = get_match_history(puuid=puuid, matches=matches)
    if not average_matches:
        print('Error: get_match_history returned nothing for seed_puuid')
        return pd.DataFrame(columns=['puuid'])
    
    average_players = set()
    for matchId in average_matches:
        match_data = get_match_data(matchId=matchId)

        if not match_data  or 'info' not in match_data: 
            print('Error: get_match_history returned nothing for seed_puuid')
            continue

        for p in match_data['info']['participants']:
            player = p.get('puuid')
            if player:
                average_players.add(player)
        time.sleep(1.2)
        
    average_player_df = pd.DataFrame(list(average_players), columns=['puuid'])
    print(f'Returning {len(average_players)} players.')
    return average_player_df

In [ ]:
#Processes the data from a match into multiple dataframes for the match metadata, team-wide data, and the multitude of stats for all players for the database
def process_match(match_json):
    metadata = match_json['metadata']
    info = match_json['info']
    matchId = match_json['metadata']['matchId']

    raw_duration = info['gameDuration']
    minutes = int(raw_duration // 60)
    seconds = int(raw_duration % 60)
    readable_time = f"{minutes}:{seconds:02d}"

    match_df = pd.DataFrame([{
        #basic match metadata
        'match_id': matchId,
        'game_version': info['gameVersion'],
        'game_duration': readable_time,
        'queue_id': info['queueId'],
        'game_mode': info['gameMode']
    }])

    teams = match_json['info']['teams']
    team_records = []
    for team in teams:
        #team-wide data for each match, mainly objectives
        obj = team['objectives']
        ban_ids = ",".join([str(ban['championId']) for ban in team.get('bans', [])])

        first_objs = [obj['baron']['first'], obj['champion']['first'], obj['dragon']['first'], obj['horde']['first'], obj['inhibitor']['first'], obj['tower']['first']]
        first_control = sum(1 for first in first_objs if first)

        objectives_taken = obj['baron']['kills'] + obj['dragon']['kills'] + obj['horde']['kills'] + obj['inhibitor']['kills'] + obj['tower']['kills']
        macro_efficiency = objectives_taken / obj['champion']['kills'] if obj['champion']['kills'] > 0 else objectives_taken

        team_records.append({
            'match_id': matchId,
            'team_id': team['teamId'],
            'win': 1 if team['win'] else 0,
            'bans': ban_ids,

            'baron_kills': obj['baron']['kills'],
            'baron_first': 1 if obj['baron']['first'] else 0,
            
            'champion_kills': obj['champion']['kills'],
            'champion_first': 1 if obj['champion']['first'] else 0,

            'dragon_kills': obj['dragon']['kills'],
            'dragon_first': 1 if obj['dragon']['first'] else 0,

            'grubs_kills': obj['horde']['kills'],
            'grubs_first': 1 if obj['horde']['first'] else 0,

            'inhibitor_kills': obj['inhibitor']['kills'],
            'inhibitor_first': 1 if obj['inhibitor']['first'] else 0,

            'tower_kills': obj['tower']['kills'],
            'tower_first': 1 if obj['tower']['first'] else 0,

            'macro_efficiency': macro_efficiency,
            'first_control': first_control
        })

    team_df = pd.DataFrame(team_records)

    participants = match_json['info']['participants']
    players_list = []
    
    for player in participants:
        #common player data for each match
        player_data = {
            'match_id': matchId,
            'puuid': player['puuid'],
            'riotIdGameName': player['riotIdGameName'],
            'riotIdTagline': player['riotIdTagline'],
            'team_id': player['teamId'],
            'win': 1 if player['win'] else 0,
            'gameEndedInEarlySurrender': 1 if player['gameEndedInEarlySurrender'] else 0,
            'gameEndedInSurrender': 1 if player['gameEndedInSurrender'] else 0,

            'championId': player['championId'],
            'championName': player['championName'],
            'teamPosition': player['teamPosition'],

            'kills': player['kills'],
            'deaths': player['deaths'],
            'assists': player['assists'],
            'summOne': player['summoner1Id'],
            'summTwo': player['summoner2Id'],
            'totalMinionsKilled': player['totalMinionsKilled'],

            'largestKillingSpree': player['largestKillingSpree'],
            'longestTimeSpentLiving': player['longestTimeSpentLiving'],
            'totalTimeSpentDead': player['totalTimeSpentDead'],

            'goldEarned': player['goldEarned'],
            'goldSpent': player['goldSpent'],

            'item0': player['item0'],
            'item1': player['item1'],
            'item2': player['item2'],
            'item3': player['item3'],
            'item4': player['item4'],
            'item5': player['item5'],
            'item6': player['item6'],

            'firstBloodKill': 1 if player['firstBloodKill'] else 0,
            'firstBloodAssist': 1 if player['firstBloodAssist'] else 0,
            'firstTowerKill': 1 if player['firstTowerKill'] else 0,
            'firstTowerAssist': 1 if player['firstTowerAssist'] else 0,
            'turretTakedowns': player['turretTakedowns'],
            'dragonKills': player['dragonKills'],

            'damageDealtToBuildings': player['damageDealtToBuildings'],
            'damageDealtToObjectives': player['damageDealtToObjectives'],
            'damageDealtToTurrets': player['damageDealtToTurrets'],

            'totalDamageDealtToChampions': player['totalDamageDealtToChampions'],
            'totalDamageTaken': player['totalDamageTaken'],
            'totalDamageShieldedOnTeammates': player['totalDamageShieldedOnTeammates'],
            'totalHeal': player['totalHeal'],
            'totalHealsOnTeammates': player['totalHealsOnTeammates'],
            'totalDamageTaken': player['totalDamageTaken'],
            'totalTimeCCDealt': player['totalTimeCCDealt'],

            'totalAllyJungleMinionsKilled': player['totalAllyJungleMinionsKilled'],
            'totalEnemyJungleMinionsKilled': player['totalEnemyJungleMinionsKilled'],

            'visionScore': player['visionScore'],
            'wardsPlaced': player['wardsPlaced'],
            'wardsKilled': player['wardsKilled'],
            'visionWardsBoughtInGame': player['visionWardsBoughtInGame'],
        }
        #extra unique challenges that may see more specific correlation with winning, using get() here as some challenges might not exist in some gamemodes and might error otherwise
        target_challenges = ['goldPerMinute', 'bountyGold', 'killParticipation', 'turretPlatesTaken', 'dragonTakedowns', 'soloKills', 'takedownsFirstXMinutes', 'takedownsFirst25Minutes', 
                             'damagePerMinute', 'earlyLaningPhaseGoldExpAdvantage', 'laneMinionsFirst10Minutes', 'laningPhaseGoldExpAdvantage', 'maxCsAdvantageOnLaneOpponent', 
                             'maxLevelLeadLaneOpponent', 'killsOnOtherLanesEarlyJungleAsLaner', 'fastestLegendary', 'enemyChampionImmobilizations', 'maxKillDeficit', 'outnumberedKills', 
                            'pickKillWithAlly', 'saveAllyFromDeath', 'soloTurretsLategame', 'junglerKillsEarlyJungle', 'killsOnLanersEarlyJungleAsJungler', 'scuttleCrabKills', 
                            'jungleCsBefore10Minutes', 'enemyJungleMonsterKills', 'junglerTakedownsNearDamagedEpicMonster', 'visionScoreAdvantageLaneOpponent', 'visionScorePerMinute', 
                            'controlWardsPlaced', 'controlWardTimeCoverageInRiverOrEnemyHalf']
        challenges = player.get('challenges', {})
        for key in target_challenges:
            player_data[key] = challenges.get(key, 0)

        players_list.append(player_data)

    players_df = pd.DataFrame(players_list)

    return match_df, team_df, players_df

In [ ]:
#Returns a json of the entire match timeline for a given match id
def get_match_timeline_json(matchId):
    link = f'https://americas.api.riotgames.com/lol/match/v5/matches/{matchId}/timeline'
    response = requests.get(link + '?api_key=' + apiKey)
    return response.json()

#Returns players stats from a match timeline at the given timestamp
def extract_timeline_stats(timeline_json, timestamp=15):
    timeframe = timeline_json['info']['frames'][timestamp]
    player_data = timeframe['participantFrames']

    stats_at_time = []
    for playerId, data in player_data.items():
        stats_at_time.append({'participantId': playerId, 'totalGold': data['totalGold'], 'level': data['level'], 'minionsKilled': data['minionsKilled'] + data['jungleMinionsKilled']})

    return stats_at_time

#Adds columns for gold, xp, minions killed, and gold difference vs lane opponent to player data frames at given timestamps. Can insert multiple timestamps at once
def process_match_with_timeline(match_json, timeline_json, timestamps = [5, 10, 15, 20]):
    match_df, team_df, players_df = process_match(match_json)

    for minute in timestamps:
        if len(timeline_json['info']['frames']) <= minute:
            continue

        stats_list = extract_timeline_stats(timeline_json, minute)
        snapshot_at_tf = {}

        for participant in stats_list:
            playerId = int(participant['participantId'])
            snapshot_at_tf[playerId] = participant

        gold_ts, xp_ts, minions_ts, gold_diff_ts = [], [], [], []

        for i in range(1, 11):
            player_stats = snapshot_at_tf.get(i, {'totalGold': 0, 'level': 0, 'minionsKilled': 0})

            opponent_id = i + 5 if i <= 5 else i - 5
            opponent_stats = snapshot_at_tf.get(opponent_id,{'totalGold': 0, 'level': 0, 'minionsKilled': 0})

            gold_ts.append(player_stats['totalGold'])
            xp_ts.append(player_stats['level'])
            minions_ts.append(player_stats['minionsKilled'])
            gold_diff_ts.append(player_stats['totalGold'] - opponent_stats['totalGold'])

        players_df[f'goldAt{minute}'] = gold_ts
        players_df[f'levelAt{minute}'] = xp_ts
        players_df[f'minionsAt{minute}'] = minions_ts
        players_df[f'goldDiffAt{minute}'] = gold_diff_ts

    for minute in timestamps:
        for stat in ['goldAt', 'levelAt', 'minionsAt', 'goldDiffAt']:
            col_name = f'{stat}{minute}'
            if col_name not in players_df.columns:
                players_df[col_name] = np.nan

    return match_df, team_df, players_df

In [ ]:
#Checks if a match id has already been processed to avoid duplicates. Makes sure the database exists first
def is_match_processed(matchId, conn):
    cursor = conn.cursor()
    try:
        cursor.execute(f"SELECT 1 FROM matches WHERE match_id = '{matchId}'")
        return cursor.fetchone() is not None
    except sql.OperationalError:
        return False

#Saves the dataframes returned by process_match() to the database
def save_to_db(match_df, team_df, players_df, db_path='league_data.db'):
    conn = sql.connect(db_path)

    try: 
        match_df.to_sql('matches', conn, if_exists='append', index=False)
        team_df.to_sql('teams', conn, if_exists='append', index=False)
        players_df.to_sql('participants', conn, if_exists='append', index=False)
        conn.commit()

    except Exception as e:

        actual_error = e.__cause__ if e.__cause__ else e
        print(f"Database Error: {actual_error}")
        conn.rollback()

    finally: conn.close()

# Wipes existing tables in the database for a fresh restart
def reset_db(db='league_data.db'):
    try:
        conn = sql.connect(db)
        reset = ['matches', 'teams', 'participants']
        for table in reset:
            conn.execute(f'DROP TABLE IF EXISTS {table}')
        conn.commit()
        conn.close()

    except Exception as e:
        print(f"Database error: {e}")

In [ ]:
#Start of main function, clears db if it exists, output of ladder shows the top players of the solo queue ladder including their puuids, wins/losses, ranked league points (LP), and if they're on a winstreak
reset_db('league_data.db')
conn = sql.connect('league_data.db')

ladder_df = get_ladder(top=20)
ladder_df

In [ ]:
#Fetch match history of match ids for COUNT games for every top player
top_match_ids = get_match_ids(ladder_df, matches=2)
display(top_match_ids)

In [ ]:
#Processes matches and saves them to league_data.db. Displays The 3 tables matches, teams, participants at the end
#Running with process_match_with_timeline over process_match to get data throughout the game instead of end-of-game only stats
intervals = [5, 10, 15, 20, 25]

for index, match_id in top_match_ids['matchId'].items():
    if is_match_processed(match_id, conn):
        continue

    try:
        match_data = get_match_data(match_id)
        participants = match_data['info']['participants']
        early_ff = any(participant.get('gameEndedInEarlySurrender', False) for participant in participants)

        if match_data['info']['gameMode'] != "CLASSIC" or early_ff:
            continue

        timeline_data = get_match_timeline_json(match_id)

        match_df, team_df, players_df = process_match_with_timeline(match_data, timeline_data, timestamps=intervals)
        match_df['rank_group'] = 'Top'
        team_df['rank_group'] = 'Top'
        players_df['rank_group'] = 'Top'

        save_to_db(match_df, team_df, players_df)
        print(f"Successfully saved match {index + 1} data to database")
        time.sleep(2.35)

    except Exception as e:
        print(f"Error processing match {match_id}: {e}")

print("Successfully saved all match data to database")

df_matches = pd.read_sql("SELECT * FROM matches", conn)
df_teams = pd.read_sql("SELECT * FROM teams", conn)
df_participants = pd.read_sql("SELECT * FROM participants", conn)

display(df_matches, df_teams, df_participants)

In [ ]:
# We also get games from more average ranks using my personal account as the foundation, defined at the top of this project.
# Idea: use these more chaotic games to compare against the refined matches from top players
# get_average_players returns a little under matches * 10 puuids, include an extra match in get_average_players or get_match_ids to make up for parties/unwanted game-modes, and get an equal sample to top players
personal_puuid = get_puuid(gameName='Shadow', tagLine='MC3')
average_players = get_average_players(puuid=personal_puuid, matches=2)
av_matches = get_match_ids(average_players, matches=3)
display(av_matches)

In [ ]:
#Like processing top matches above, this also processes the average matches and saves them to league_data.db. Displays The 3 tables matches, teams, participants at the end
#Running with process_match_with_timeline over process_match to get data throughout the game instead of end-of-game only stats
for index, match_id in av_matches['matchId'].items():
    if is_match_processed(match_id, conn):
        continue
    try:
        match_data = get_match_data(match_id)
        if 'info' not in match_data:
            print(f'Error for {match_id}: {match_data}')
            continue

        participants = match_data['info']['participants']
        early_ff = any(participant.get('gameEndedInEarlySurrender', False) for participant in participants)

        if match_data['info']['gameMode'] != "CLASSIC" or early_ff:
            continue

        timeline_data = get_match_timeline_json(match_id)

        match_df, team_df, players_df = process_match_with_timeline(match_data, timeline_data, timestamps=intervals)
        match_df['rank_group'] = 'Average'
        team_df['rank_group'] = 'Average'
        players_df['rank_group'] = 'Average'

        save_to_db(match_df, team_df, players_df)
        print(f"Successfully saved match {index + 1} data to database")
        time.sleep(2.35)

    except Exception as e:
        print(f"Error processing match {match_id}: {e}")

print("Successfully saved all match data to database")

df_matches = pd.read_sql("SELECT * FROM matches WHERE rank_group = 'Average'", conn)
df_teams = pd.read_sql("SELECT * FROM teams WHERE rank_group = 'Average'", conn)
df_participants = pd.read_sql("SELECT * FROM participants WHERE rank_group = 'Average'", conn)

display(df_matches, df_teams, df_participants)

In [ ]:
# With us now having of the top players data, find trends/correlations with wins
# Get numeric team data, find trends/correlations with team features and wins. Create data visualization
top_teams_df = pd.read_sql("SELECT * from teams WHERE rank_group = 'Top'", conn)
av_teams_df = pd.read_sql("SELECT * from teams WHERE rank_group = 'Average'", conn)

top_teams_df['is_blue_side'] = (top_teams_df['team_id'] == 100).astype(int)
av_teams_df['is_blue_side'] = (av_teams_df['team_id'] == 100).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

top_teams = top_teams_df.select_dtypes(include='number').corr()['win'].drop(['win', 'team_id'])
av_teams = av_teams_df.select_dtypes(include='number').corr()['win'].drop(['win', 'team_id'])

team_comparison = pd.DataFrame({'Top Players': top_teams, 'Average Player': av_teams}).sort_values(by='Top Players', ascending=True)
team_comparison.plot(kind='barh', ax=axes[0], color=['blue', 'red'], edgecolor='black')

axes[0].set_title(f'Team-wide Feature Correlation with Winning: By Rank')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].grid(axis='x', linestyle='--')
axes[0].set_ylabel('Team Features')

full_teams = pd.concat([top_teams_df, av_teams_df])
full_teams_corr = full_teams.select_dtypes(include='number').corr()['win'].drop(['win', 'team_id']).sort_values(ascending=True)

full_teams_corr.plot(kind='barh', ax=axes[1], color='purple', edgecolor='black')
axes[1].set_title(f'Team-wide Feature Correlation with Winning: Full Correlation')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].grid(axis='x', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Get numeric player data, find trends/correlations with player oriented features and wins. Create data visualization
# Top players
top_players_df = pd.read_sql("SELECT * from participants WHERE rank_group = 'Top'", conn)
av_players_df = pd.read_sql("SELECT * from participants WHERE rank_group = 'Average'", conn)

drop_ids = ['win', 'item0', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'summOne', 'summTwo', 'championId', 'team_id', 'gameEndedInSurrender', 'gameEndedInEarlySurrender', 'takedownsFirst25Minutes']

fig, axes = plt.subplots(1, 2, figsize=(24, 16))

top_players_df['win'] = top_players_df['win'].astype(int)
top_numeric_corr = top_players_df.select_dtypes(include='number').corr()['win'].drop(drop_ids)

av_players_df['win'] = av_players_df['win'].astype(int)
av_numeric_corr = av_players_df.select_dtypes(include='number').corr()['win'].drop(drop_ids)
corr_matrix= pd.DataFrame({'Top Players': top_numeric_corr, 'Average Player': av_numeric_corr})
significant_corrs = corr_matrix.sort_values(by='Top Players', ascending=True)

significant_corrs.plot(kind='barh', ax=axes[0], color=['blue', 'red'], edgecolor='black')
axes[0].set_title(f'Feature Correlation with Winning: Rank Comparison')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].grid(True)
axes[0].set_ylabel('Player Features')

full_players = pd.concat([top_players_df, av_players_df])
full_players_corr = full_players.select_dtypes(include='number').corr()['win'].drop(drop_ids).sort_values(ascending=True)

full_players_corr.plot(kind='barh', ax=axes[1], color='purple', edgecolor='black')
axes[1].set_title(f'Feature Correlation with Winning: Full Correlation')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Use snapshots of total gold leads at timestamps to find if there is a correlation between gold leads at and wins as time goes on
# Find what point in the game gold leads matter most. Create data visualization
fig, axes = plt.subplots(2, len(intervals), figsize=(30, 12), sharey=True)
if(len(intervals) == 1): axes = [axes]

for index, minute in enumerate(intervals):
    ax_top = axes[0, index]
    ax_bot = axes[1, index]

    top_players_df[f'goldRangeAt{minute}'] = pd.cut(top_players_df[f'goldDiffAt{minute}'], bins=range(-5000, 5500, 500), right=True)
    top_win_analysis = top_players_df.groupby(f'goldRangeAt{minute}')['win'].mean()

    av_players_df[f'goldRangeAt{minute}'] = pd.cut(av_players_df[f'goldDiffAt{minute}'], bins=range(-5000, 5500, 500), right=True)
    av_win_analysis = av_players_df.groupby(f'goldRangeAt{minute}')['win'].mean()

    win_analysis = pd.DataFrame({'Top Players': top_win_analysis, "Average Players": av_win_analysis})
    win_analysis.plot(kind='bar', ax=ax_top, edgecolor='black', color=['blue', 'red'])

    combined_df = pd.concat([top_players_df, av_players_df])
    combined_df[f'goldRangeAt{minute}'] = pd.cut(combined_df[f'goldDiffAt{minute}'], bins=range(-5000, 5500, 500), right=True)
    combined_analysis = combined_df.groupby(f'goldRangeAt{minute}')['win'].mean()
    combined_analysis.plot(kind='bar', ax=ax_bot, edgecolor='black', color='purple')

    ax_top.set_title(f'Win % per Gold Range vs Lane Opponent @ Min {minute}')
    ax_bot.set_title(f'Win % per Gold Range vs Lane Opponent @ Min {minute}: Combined Data')

    for ax in [ax_top, ax_bot]:
        ax.set_ylabel('Win Probability')
        ax.set_xlabel('Gold Differential')
        ax.axhline(.5, color='black', linestyle='--', alpha=.5)
        ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create dynamic columns to measure the momentum of a game within specific timeframes, and if leads snowball into wins
# Find what point in the game gold leads matter most. Create data visualization
differential_cols = [f'goldDiffAt{interval}' for interval in intervals]
momentum_cols = []

for df in [top_players_df, av_players_df]:
    for i in range(len(intervals) - 1):
        start = intervals[i]
        end = intervals[i+1]
        col = f'momentum_{start}To{end}'

        if col not in momentum_cols:
            momentum_cols.append(col)
        df[col] = df[f'goldDiffAt{end}'] - df[f'goldDiffAt{start}']

fig, axes = plt.subplots(1, 4, figsize=(20,4))

top_static = top_players_df[differential_cols + ['win']].corr()['win'].drop('win')
av_static = av_players_df[differential_cols + ['win']].corr()['win'].drop('win')
static_comparison = pd.DataFrame({'Top Players': top_static, 'Average Player': av_static})

top_dynamic = top_players_df[momentum_cols + ['win']].corr()['win'].drop('win')
av_dynamic = av_players_df[momentum_cols + ['win']].corr()['win'].drop('win')
dynamic_comparison = pd.DataFrame({'Top Players': top_dynamic, 'Average Players': av_dynamic})

combined_data = pd.concat([top_players_df, av_players_df], ignore_index=True)

full_static = combined_data[differential_cols + ['win']].corr()['win'].drop('win').sort_values(ascending=True)
full_dynamic = combined_data[momentum_cols + ['win']].corr()['win'].drop('win').sort_values(ascending=True)

dfs = [static_comparison, dynamic_comparison, full_static, full_dynamic]
colors = [['blue', 'red'], ['blue', 'red'], 'purple', 'purple']

for i in range(4):
    dfs[i].plot(kind='barh', ax=axes[i], color=colors[i], edgecolor='black')

    axes[i].set_xlabel(f'Correlation Coefficient')
    axes[i].grid(axis='x')

    if i % 2 == 0:
        axes[i].set_title(f'Static Gold Lead')
        axes[i].set_ylabel(f'Gold Intervals')
        axes[i].set_title(f'Static Gold Lead')
    else:
        axes[i].set_title(f'Momentum/Snowballing')
        axes[i].set_ylabel(f'Momentum Intervals')

plt.tight_layout()
plt.show()

In [ ]:
# Create a prediction model to see if we can predict if a game is a win given a set of stats
X = full_teams.select_dtypes(include='number').drop(columns=['win', 'team_id'])
y = full_teams['win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
confusion_matrix = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print(f'F1-score: {f1:.4f}')
print('Confusion Matrix:\n', confusion_matrix)

In [ ]:
conn.close()